# 19장 텍스트 데이터 — 글자를 숫자로

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `fetch_statutes.py` 로 데이터를 받고 `p01`~`p03`, `statutes_dataset.py` 를 치고 실행 | [ch19_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `Vocab.encode` 와 `BagOfWords.forward` 각 줄 뒤에 shape을 주석으로 | [ch19_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `StatuteDataset` 복원. 그다음 14장 패키지의 `data/` 에 `statutes.py` 로 넣고 `--dataset statutes --model bow` 로 학습되게 등록 | [ch19_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "19"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/19")

준비 끝. 작업 폴더: code/19


## `fetch_statutes.py` — 19.0 텍스트 실습 데이터 만들기: 국가법령정보 OPEN API 에서 법령 조문을 받아 텍스트 파일로 저장한다.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

**이 파일은 실행하지 않아도 됩니다.** 받아 둔 조문 파일이 저장소의 `data/statutes/` 에 들어 있습니다. 직접 받아 보려면 open.law.go.kr 에서 OPEN API 이용 신청을 하고 발급된 ID 를 환경변수 `LAW_OC` 에 넣은 뒤 실행합니다.

```python
# 19.0 텍스트 실습 데이터 만들기: 국가법령정보 OPEN API 에서 법령 조문을 받아 텍스트 파일로 저장한다.
# 법령·판결문은 저작권법 제7조에 따라 저작권 보호를 받지 않으므로 실습 데이터로 자유롭게 쓸 수 있다.
# 준비: https://open.law.go.kr 에서 OPEN API 이용 신청 → 발급된 ID 를 환경변수 LAW_OC 에 넣는다.
# 실행: python fetch_statutes.py            → JUCOS_DATA/statutes/{법령명}.txt (한 줄 = 조문 하나)
import json
import os
import re
import sys
import time
import urllib.parse
import urllib.request

OC = os.environ.get("LAW_OC")
OUT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "statutes")
LAWS = ["민법", "형법", "민사소송법", "형사소송법", "상법", "행정소송법"]
HEADERS = {"User-Agent": "Mozilla/5.0", "Referer": "https://www.law.go.kr/"}   # Referer 가 없으면 거부된다


def get_json(url):
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read().decode("utf-8"))


def find_mst(name):
    """법령명으로 검색해 현행 법령의 일련번호(MST)를 얻는다."""
    q = urllib.parse.urlencode({"OC": OC, "target": "law", "type": "JSON", "query": name, "display": 100, "sort": "lasc"})
    data = get_json(f"https://www.law.go.kr/DRF/lawSearch.do?{q}")
    laws = data.get("LawSearch", {}).get("law", [])
    if isinstance(laws, dict):
        laws = [laws]
    exact = [l for l in laws if l.get("법령명한글") == name and l.get("현행연혁코드") == "현행"]
    if not exact:
        raise RuntimeError(f"{name}: 검색 결과 없음 ({[l.get('법령명한글') for l in laws][:5]})")
    return exact[0]["법령일련번호"], exact[0].get("시행일자")


def strip_tags(s):
    return re.sub(r"<[^>]+>", "", s or "").replace("\r", "").strip()


def fetch_articles(mst):
    """조문 본문을 (조문번호, 제목, 내용) 목록으로."""
    q = urllib.parse.urlencode({"OC": OC, "target": "law", "type": "JSON", "MST": mst})
    data = get_json(f"https://www.law.go.kr/DRF/lawService.do?{q}")
    units = data["법령"]["조문"]["조문단위"]
    if isinstance(units, dict):
        units = [units]
    out = []
    for u in units:
        if u.get("조문여부") != "조문":            # 장·절 제목 등은 건너뛴다
            continue
        body = strip_tags(u.get("조문내용", ""))
        # 항이 따로 있으면 이어 붙인다
        hangs = u.get("항", [])
        if isinstance(hangs, dict):
            hangs = [hangs]
        for h in hangs:
            body += " " + strip_tags(h.get("항내용", ""))
            hos = h.get("호", [])
            if isinstance(hos, dict):
                hos = [hos]
            for ho in hos:
                body += " " + strip_tags(ho.get("호내용", ""))
        body = re.sub(r"\s+", " ", body).strip()
        if body:
            out.append((u.get("조문번호"), u.get("조문제목") or "", body))
    return out


def main():
    if not OC:
        sys.exit("환경변수 LAW_OC 가 없습니다. open.law.go.kr 에서 발급받은 ID 를 넣으세요.")
    os.makedirs(OUT, exist_ok=True)
    for name in LAWS:
        if os.path.exists(os.path.join(OUT, f"{name}.txt")):
            print(f"{name:<8} 이미 있음 — 건너뜀")
            continue
        mst, eff = find_mst(name)
        arts = fetch_articles(mst)
        path = os.path.join(OUT, f"{name}.txt")
        with open(path, "w", encoding="utf-8") as f:
            for no, title, body in arts:
                f.write(body + "\n")
        chars = sum(len(a[2]) for a in arts)
        print(f"{name:<8} MST {mst}  시행 {eff}  조문 {len(arts):5d}개  {chars:8,}자  → {path}")
        time.sleep(1)


if __name__ == "__main__":
    main()
```

## `p01_tokenize.py` — 19.1~19.2 텍스트 → 숫자: 토큰화, 어휘 사전, 패딩

In [ ]:
# 19.1~19.2 텍스트 → 숫자: 토큰화, 어휘 사전, 패딩
import os
from collections import Counter
import torch

DATA = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "statutes")


def load_lines(name):
    with open(os.path.join(DATA, f"{name}.txt"), encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]


civil = load_lines("민법")
crim = load_lines("형법")
print("민법 조문", len(civil), "개 | 형법 조문", len(crim), "개")
print("예:", civil[0][:60])

민법 조문 1193 개 | 형법 조문 402 개
예: 제1조(법원) 민사에 관하여 법률에 규정이 없으면 관습법에 의하고 관습법이 없으면 조리에 의한다.


In [ ]:
# ---- 1) 토큰화: 문장을 조각(토큰)으로. 가장 단순한 방법 둘 — 글자 단위, 공백 단위
s = civil[3]
print("\n원문     :", s)
print("글자 토큰:", list(s)[:20], "… 총", len(s))
print("공백 토큰:", s.split()[:10], "… 총", len(s.split()))


원문     : 제4조(성년) 사람은 19세로 성년에 이르게 된다.
글자 토큰: ['제', '4', '조', '(', '성', '년', ')', ' ', '사', '람', '은', ' ', '1', '9', '세', '로', ' ', '성', '년', '에'] … 총 28
공백 토큰: ['제4조(성년)', '사람은', '19세로', '성년에', '이르게', '된다.'] … 총 6


In [ ]:
# ---- 2) 어휘 사전: 토큰 → 번호. 자주 나오는 순으로 번호를 준다
PAD, UNK = "<pad>", "<unk>"                     # 특수 토큰: 빈칸 채우기, 모르는 토큰


def build_vocab(sentences, tokenizer, min_count=2, max_size=5000):
    counter = Counter(tok for s in sentences for tok in tokenizer(s))
    tokens = [t for t, c in counter.most_common(max_size) if c >= min_count]
    itos = [PAD, UNK] + tokens                  # index → token
    stoi = {t: i for i, t in enumerate(itos)}   # token → index (3장의 딕셔너리)
    return itos, stoi


tokenize = lambda s: s.split()
itos, stoi = build_vocab(civil + crim, tokenize)
print("\n어휘 크기:", len(itos), "| 상위 10:", itos[2:12])
print("'권리' →", stoi.get("권리"), "| '없는말' →", stoi.get("없는말", stoi[UNK]))


def encode(s, stoi, tokenizer, max_len):
    ids = [stoi.get(t, stoi[UNK]) for t in tokenizer(s)][:max_len]     # 자르고
    return ids + [stoi[PAD]] * (max_len - len(ids))                     # 채운다 (패딩)


어휘 크기: 4017 | 상위 10: ['그', '또는', '수', '때에는', '있다.', '한다.', '이하의', '경우에는', '처한다.', '있는']
'권리' → 2360 | '없는말' → 1


In [ ]:
# ---- 3) 길이 맞추기: 문장마다 길이가 다르므로 max_len 으로 자르거나 <pad> 로 채운다
for s in (civil[3], civil[0]):
    ids = encode(s, stoi, tokenize, max_len=12)
    print(f"\n{s[:40]}…")
    print("  ids :", ids)
    print("  복원:", [itos[i] for i in ids])


제4조(성년) 사람은 19세로 성년에 이르게 된다.…
  ids : [1, 145, 1, 2492, 102, 288, 0, 0, 0, 0, 0, 0]
  복원: ['<unk>', '사람은', '<unk>', '성년에', '이르게', '된다.', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']

제1조(법원) 민사에 관하여 법률에 규정이 없으면 관습법에 의하고 관습법…
  ids : [1, 1, 68, 219, 287, 134, 2487, 2488, 1, 134, 1, 127]
  복원: ['<unk>', '<unk>', '관하여', '법률에', '규정이', '없으면', '관습법에', '의하고', '<unk>', '없으면', '<unk>', '의한다.']


In [ ]:
# ---- 4) 배치: 이제 텐서로 쌓을 수 있다 (15장의 DataLoader 가 하던 일)
batch = torch.tensor([encode(s, stoi, tokenize, 12) for s in civil[:4]])
print("\n배치 shape:", batch.shape, "dtype:", batch.dtype)          # [4, 12] int64


배치 shape: torch.Size([4, 12]) dtype: torch.int64


In [ ]:
# ---- 5) 조문 길이 분포: max_len 을 얼마로 잡을지 데이터가 말해 준다
lens = sorted(len(tokenize(s)) for s in civil + crim)
print(f"\n토큰 길이: 최소 {lens[0]}  중앙값 {lens[len(lens)//2]}  90% {lens[int(len(lens)*0.9)]}  최대 {lens[-1]}")


토큰 길이: 최소 2  중앙값 22  90% 49  최대 319


## `p02_embedding.py` — 19.3 임베딩: 번호(정수)를 벡터(실수 여러 개)로. 학습되는 조회표.

In [ ]:
# 19.3 임베딩: 번호(정수)를 벡터(실수 여러 개)로. 학습되는 조회표.
import torch
import torch.nn as nn

torch.manual_seed(0)
vocab_size, dim = 10, 4
emb = nn.Embedding(vocab_size, dim)                   # 10개 토큰 각각에 4차원 벡터
print(emb.weight.shape)                               # [10, 4]: 조회표 자체가 파라미터

ids = torch.tensor([[3, 1, 0, 0],                     # 문장 2개, 길이 4 (0은 <pad>)
                    [5, 3, 9, 1]])
vec = emb(ids)
print("입력", tuple(ids.shape), "→ 출력", tuple(vec.shape))   # [2, 4] → [2, 4, 4] = [배치, 길이, 차원]
print("토큰 3 의 벡터:", [round(v, 3) for v in emb.weight[3].tolist()])
print("같은 토큰은 어디서나 같은 벡터:", torch.equal(vec[0, 0], vec[1, 1]))

torch.Size([10, 4])
입력 (2, 4) → 출력 (2, 4, 4)
토큰 3 의 벡터: [0.12, 1.238, 1.117, -0.247]
같은 토큰은 어디서나 같은 벡터: True


In [ ]:
# 문장 벡터 만들기 (가장 단순한 방법): 토큰 벡터의 평균. <pad> 는 빼고.
mask = (ids != 0).unsqueeze(-1)                       # [2, 4, 1] 패딩이 아닌 자리만 1
sent = (vec * mask).sum(1) / mask.sum(1)              # [2, 4]
print("문장 벡터 shape:", tuple(sent.shape))

문장 벡터 shape: (2, 4)


In [ ]:
# 이것이 곧 분류기: 임베딩 → 평균 → 선형층. RNN 도 Transformer 도 "평균" 자리를 더 좋은 것으로 바꾼 것이다.
clf = nn.Sequential(nn.Linear(dim, 2))
print("분류 점수:", tuple(clf(sent).shape))

분류 점수: (2, 2)


## `statutes_dataset.py` — 19.4 텍스트 Dataset: 조문 → (토큰 번호 텐서, 법령 번호). 15장의 ImageFolderDataset 과 같은 뼈대.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 19.4 텍스트 Dataset: 조문 → (토큰 번호 텐서, 법령 번호). 15장의 ImageFolderDataset 과 같은 뼈대.
import os
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader

DATA = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "statutes")
PAD, UNK = "<pad>", "<unk>"


def tokenize(s):
    return s.split()


class Vocab:
    def __init__(self, sentences, min_count=2, max_size=8000):
        counter = Counter(t for s in sentences for t in tokenize(s))
        self.itos = [PAD, UNK] + [t for t, c in counter.most_common(max_size) if c >= min_count]
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, s, max_len):
        ids = [self.stoi.get(t, 1) for t in tokenize(s)][:max_len]
        return ids + [0] * (max_len - len(ids))


class StatuteDataset(Dataset):
    """여러 법령의 조문을 모아 '어느 법령의 조문인가' 를 맞히는 분류 데이터셋."""

    def __init__(self, laws, split="train", vocab=None, max_len=40, test_ratio=0.2, seed=0):
        self.classes = list(laws)
        items = []
        for label, name in enumerate(laws):
            with open(os.path.join(DATA, f"{name}.txt"), encoding="utf-8") as f:
                lines = [l.strip() for l in f if l.strip()]
            lines = [l for l in lines if len(tokenize(l)) >= 4]     # '제786조 삭제' 같은 빈 조문은 뺀다
            items += [(l, label) for l in lines]
        g = torch.Generator().manual_seed(seed)             # 같은 씨앗이면 같은 분할
        perm = torch.randperm(len(items), generator=g).tolist()
        n_test = int(len(items) * test_ratio)
        idx = perm[n_test:] if split == "train" else perm[:n_test]
        self.items = [items[i] for i in idx]
        self.vocab = vocab or Vocab([s for s, _ in self.items])   # 어휘는 학습 데이터로만 만든다
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        text, label = self.items[i]
        return torch.tensor(self.vocab.encode(text, self.max_len)), label


if __name__ == "__main__":
    laws = ["민법", "형법", "민사소송법", "형사소송법"]
    train = StatuteDataset(laws, "train")
    test = StatuteDataset(laws, "test", vocab=train.vocab)        # 시험 데이터는 학습 어휘를 그대로 쓴다
    print("학습", len(train), "시험", len(test), "| 어휘", len(train.vocab), "| 클래스", train.classes)
    x, y = train[0]
    print("train[0] ->", tuple(x.shape), x.dtype, "| label", y, train.classes[y])
    print("  원문:", train.items[0][0][:50])
    print("  복원:", " ".join(train.vocab.itos[i] for i in x.tolist() if i != 0)[:50])
    xb, yb = next(iter(DataLoader(train, batch_size=8, shuffle=True)))
    print("배치:", tuple(xb.shape), tuple(yb.shape))
    counts = Counter(train.classes[l] for _, l in train.items)
    print("학습 클래스별:", dict(counts))
```

In [ ]:
!python statutes_dataset.py

학습 2108 시험 526 | 어휘 5986 | 클래스 ['민법', '형법', '민사소송법', '형사소송법']
train[0] -> (40,) torch.int64 | label 2 민사소송법
  원문: 제105조(소송의 총비용에 대한 재판) 상급법원이 본안의 재판을 바꾸는 경우 또는 사건을 
  복원: <unk> 총비용에 대한 재판) 상급법원이 본안의 재판을 바꾸는 경우 또는 사건을 환송받거
배치: (8, 40) (8,)
학습 클래스별: {'민사소송법': 411, '민법': 896, '형사소송법': 486, '형법': 315}


## `p03_train_bow.py` — 19.5 첫 텍스트 분류기: 임베딩 → 평균 → 선형층 ("단어 주머니" 모델). 12장 루프 그대로.

In [ ]:
# 19.5 첫 텍스트 분류기: 임베딩 → 평균 → 선형층 ("단어 주머니" 모델). 12장 루프 그대로.
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from statutes_dataset import StatuteDataset

LAWS = ["민법", "형법", "민사소송법", "형사소송법"]
torch.manual_seed(0)
train_set = StatuteDataset(LAWS, "train")
test_set = StatuteDataset(LAWS, "test", vocab=train_set.vocab)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256)


class BagOfWords(nn.Module):
    def __init__(self, vocab_size, dim, num_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim, padding_idx=0)   # <pad> 의 벡터는 0 으로 고정
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, ids):                                       # ids: [B, L]
        vec = self.emb(ids)                                       # [B, L, dim]
        mask = (ids != 0).unsqueeze(-1).float()                   # 패딩 제외 평균
        sent = (vec * mask).sum(1) / mask.sum(1).clamp(min=1)     # [B, dim]
        return self.fc(sent)


model = BagOfWords(len(train_set.vocab), dim=64, num_classes=len(LAWS))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def accuracy():
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_set)


print(f"학습 {len(train_set)} / 시험 {len(test_set)} / 어휘 {len(train_set.vocab)} / 파라미터 {sum(p.numel() for p in model.parameters()):,}")
print(f"학습 전 정확도: {accuracy():.3f}")
t0 = time.time()
for epoch in range(1, 11):
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    if epoch % 2 == 0:
        print(f"에포크 {epoch:2d}: 손실 {loss.item():.4f}  시험 정확도 {accuracy():.3f}")
print(f"경과 {time.time() - t0:.0f}초")

학습 2108 / 시험 526 / 어휘 5986 / 파라미터 383,364
학습 전 정확도: 0.183


에포크  2: 손실 1.2021  시험 정확도 0.460


에포크  4: 손실 0.8937  시험 정확도 0.546


에포크  6: 손실 0.7188  시험 정확도 0.681


에포크  8: 손실 0.4168  시험 정확도 0.778


에포크 10: 손실 0.5235  시험 정확도 0.827
경과 2초


In [ ]:
# 어느 조문을 어느 법으로 보는가: 시험 데이터 몇 개
model.eval()
with torch.no_grad():
    for i in range(5):
        x, y = test_set[i]
        pred = model(x.unsqueeze(0)).argmax(1).item()
        mark = "" if pred == y else "  ← 틀림"
        print(f"  [{LAWS[y]} → 예측 {LAWS[pred]}]{mark} {test_set.items[i][0][:45]}…")

  [민법 → 예측 민법] 제1056조(상속인없는 재산의 청산) ①제1053조제1항의 공고있은 날로부터 3월…
  [민법 → 예측 민법] 제319조(준용규정) 제213조, 제214조, 제216조 내지 제244조의 규정은…
  [형사소송법 → 예측 민법]  ← 틀림 제354조(상소포기 후의 재상소의 금지) 상소를 취하한 자 또는 상소의 포기나 취…
  [형사소송법 → 예측 형사소송법] 제163조(당사자의 참여권, 신문권) ①검사, 피고인 또는 변호인은 증인신문에 참…
  [민법 → 예측 민법] 제570조(동전-매도인의 담보책임) 전조의 경우에 매도인이 그 권리를 취득하여 매…


## 다음 단계

- 2단계 「주석 달기」: [ch19_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch19_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.